This notebook is used to test out ways to extract the AS train file blobs.

In [1]:
import os
import zipfile as zf
import argparse

import numpy as np
import h5py
import librosa as lb
import torchaudio
import torch

import utils.config as config
import utils.utility_funcs as my_utils

In [2]:
# Misbehaves in notebooks, works in .py files
# Initialize an argument parser to relay filenames on the command line
"""
parser = argparse.ArgumentParser(description="A script to extract Audioset files " \
"from a directory containing several blob files and extracting to destination directory based on an " \
"external text file of valid filenames. Archive's file structure is ignored in the process.")
parser.add_argument("--blob_dir", help="Path of the blob directory")
parser.add_argument("--valid_filenames", help="Text file containing valid filenames separated by '\n' ")
parser.add_argument("--target_dir", help="The destination of the extracted files")

args = parser.parse_args()
"""

'\nparser = argparse.ArgumentParser(description="A script to extract Audioset files " "from a directory containing several blob files and extracting to destination directory based on an " "external text file of valid filenames. Archive\'s file structure is ignored in the process.")\nparser.add_argument("--blob_dir", help="Path of the blob directory")\nparser.add_argument("--valid_filenames", help="Text file containing valid filenames separated by \'\n\' ")\nparser.add_argument("--target_dir", help="The destination of the extracted files")\n\nargs = parser.parse_args()\n'

In [2]:
# Read in the desired filenames
PATH_TO_AS_TRAIN_FNAMES = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\audioset_train_filenames_top50.txt'
valid_filenames = set()
with open(PATH_TO_AS_TRAIN_FNAMES, 'r') as fnames:
    for line in fnames.readlines():
        valid_filenames.add(line.rstrip('\n'))

In [3]:
hdf5_filename = "cl_data.hdf5"
group_name = "audioset_strong_train"

# Creating the HDF5 file, and the group
with (h5py.File(hdf5_filename, "a")) as f:
    group = f.create_group(group_name)


In [16]:
# Read in the file:multihot label information
audioset_train_labels = my_utils.pickle_load("pickle_objects/audioset_train_labels_50_dict.pckl")
counter = 0
for file in audioset_train_labels.keys():
    print(file)
    counter += 1
    if counter == 5:
        break

b0RFKhbpFJA
4PPmyY_-YrA
XMl9lI7mKsM
O35jXasNYxc
s451ci8Fric


In [ ]:
# These same files are found on a computing cluster, testing locally first 
PATH_TO_LOCAL_AUDIOSET_00_ZIP = r'C:\Users\mp431591\Documents\audioset_unbalanced_test\unbalanced_train_segments_part00_full.zip'
PATH_TO_TEST_DEST = r'C:\Users\mp431591\Documents\audioset_unbalanced_test\test_dest'
PATH_TO_BLOB_DIR = r'C:\Users\mp431591\Documents\audioset_unbalanced_test\faux_blob_folder_for_testing'

MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(sample_rate=config.sample_rate,
                                                     n_fft=config.n_fft,
                                                     hop_length=config.hop_length,
                                                     n_mels=config.n_mels,
                                                     f_min=config.fmin,
                                                     f_max=config.fmax)

# Open the hdf5 file, remember to close
hdf5 = h5py.File(hdf5_filename, 'a')

# Open dir full of blobs/zip files
with os.scandir(PATH_TO_BLOB_DIR) as blob_dir:

    # For each blob
    for entry in blob_dir:

        with zf.ZipFile(entry.path, 'r') as archive:
            archive_path = archive.filename

            for zip_info in archive.infolist()[0:18]:

                if zip_info.is_dir():
                    continue
                # Break archive directory structures
                zip_info.filename = os.path.basename(zip_info.filename) 
                filename = zip_info.filename
                clean_file_name = filename.rstrip(".wav")

                if clean_file_name in valid_filenames:
                    # If extracting
                    #archive.extract(zip_info, path=PATH_TO_TEST_DEST)

                    # If processing
                    with archive.open(zip_info, 'r') as audio_file:
                        target_sr = config.sample_rate
                        audio, sr = torchaudio.load(audio_file,)
                        if sr != target_sr:
                            print(f"Resampling {clean_file_name}")
                            audio = torchaudio.functional.resample(audio, 
                                                           orig_freq=sr,
                                                           new_freq=target_sr)
                        audio = my_utils.pad_or_truncate(audio, target_sr * 10)
                        mel_specgram = MEL_TRANSFORM(audio)
                        
                        mel_specgram = torch.transpose(mel_specgram, 2, 1)
                        mel_specgram = torch.log(mel_specgram + torch.finfo(torch.float32).eps)

                        hdf_path = group_name + '/' + clean_file_name
                        hdf5[hdf_path] = mel_specgram.numpy()
                        # Meta table filenames are without Ys and filenames are with Ys
                        clean_file_name_wo_Y = clean_file_name[1::]
                        hdf5[hdf_path].attrs['label'] = audioset_train_labels[clean_file_name_wo_Y]

hdf5.close()




-iKcpbSSIEc
-HEGr-l41WU
-SahGP1aKbs
-ZQXEQp3awU
-MP30EIKEdU
-9q6TWP2jEo
-jCjwUJU0aY
-W9680BebMk
-A3cOSeUcLM
-kaU-fjT8C8
-5mNXYRyIJE
-5A9F8Ak1gQ
-OBQFp6wz48
-HFITphzeVY
-AV028hx__I
-MCjiOQ0MKU
-BdPW1c_3_8
-Xvh80apkN4
-KI85vgUpdI
-RpohRXb79E
-iwmfVbmIl4
-3Qyf1jECX0
-bDW3xVC7gs
--PMnfMhn1A
-daiRP7v0s0
-03pwk56EKM
-6XGcz56P9s
-NRz46gYVoQ
-H2I8vcryHA
-GulJdbXIJA
-HeQfwKbFzg
-Wdl7u29-O4
-3781iiKiaY
-mlj06Uyuec
-5jTPNu2Unc
-gzDQxMhKsQ
-JMlwL22ldM
-Z0OfTAK5VU
-93C22DxVD4
-glN59TmfME
-iijak82s6k
-UolgqlTe2I
-kCWX1wjibc
-M7t3m9uSJA
-NOT-Kx5mKo
-7SwhiYPNkI
-CrQG7NFSMg
-Bm0VdPhIEI
-eUTaZUubYs
-Yhxm76eRuk
-fqADYDY2j8
-4qrqZ35A8A
-OkxMdTRIFs
-iQlvp-lsY4
-EyZiVzzYf4
-n-CtnbxIWg
-eEugg6wAio
-chHkc6XL5A
-g7M0w2Ugm0
-UJ03wirapo
-iOSktA4b14
-ktfcBe_gBA
-bkBuT1K9_c
-NkgiV-1kf4
-2kJ_gyPt_M
-Wzvt2yCGQg
-01YByaybkc
-CzAjJI8PbA
-18-DJ76IaQ
-5UvXobuBos
-Jhk1PJ2JoA
-Jnns0s8q2g
-eHidmQUa4I
-EssVrOkSwY
-WX9P6QAYKI
-F2lk9A8B8M
-714doUQec4
-M1qZu0rYvY
-RHewf_0KHM
-Qa9Fd-1S2A
-8LaFV1qoPc
-_sYimj4Ml4
-eQkuW-SGkA
-lW-

OSError: Unable to create link (name already exists)

In [5]:
# Inspecting the hdf5 file

def print_name(name):
    print(name)

PATH_TO_HDF5 = "cl_data.hdf5"
group_name = "audioset_strong_train_50"
with h5py.File(PATH_TO_HDF5, 'r') as hdf5:
    tmp = hdf5[group_name]
    counter = 0
    print(len(tmp.keys()))
    for name in hdf5[group_name]:
        if counter == 5:
            break
        print(tmp[name].name)
        audio = tmp[name][0] # Access the audio, since there is only member per dataset
        print(torch.from_numpy(audio).unsqueeze(0).shape) # Modify to shape it was
        print(tmp[name].attrs['label'])
        counter += 1
        

84
/audioset_strong_train_50/Y-HEGr-l41WU
torch.Size([1, 1001, 64])
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 0 0 1 0]
/audioset_strong_train_50/Y-SahGP1aKbs
torch.Size([1, 1001, 64])
[1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 0 0 0 0 0 0 0]
/audioset_strong_train_50/Y-ZQXEQp3awU
torch.Size([1, 1001, 64])
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
/audioset_strong_train_50/Y-iKcpbSSIEc
torch.Size([1, 1001, 64])
[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0]
/audioset_strong_train_50/Y0T2gqhn5uNQ
torch.Size([1, 1001, 64])
[0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 1 0 0 0]
